# Smart City Autonomous Driving — 3 AI Models Multi-Task Integration

This notebook executes real-time **Smart City Autonomous Driving** on JetRacer, integrating **3 Trained AI Models** simultaneously:

| Model | File | Task |
|-------|------|------|
| **1. YOLO Sign Detector** | `models/urban_traffic/best.onnx` | Detects traffic lights (`red-light`, `green-light`) and signs (`left-turn-sign`, `right-turn-sign`, `straight-ahead-sign`, `prohibition-sign`). |
| **2. MobileNet Collision Avoidance** | `models/urban_traffic/best_model_mobilenet.onnx` | Classifies road obstacle status (`FREE` vs `BLOCKED`). |
| **3. Conditioned Trajectory Model** | `models/urban_traffic/best_trajectory_mobilenet.onnx` | Predicts 5 route waypoints `(x, y)` given command (`LEFT`, `RIGHT`, `STRAIGHT`). |

### Processing Pipeline:
1. **`CameraStream`**: Thread-safe ROS Camera Topic Subscriber (`/csi_cam_0/image_raw`).
2. **`YOLOProcessor` + `TrafficFSM`**: Spatial filtering (BBox area & ROI) + Priority sign evaluation.
3. **`RoadProcessor`**: Realtime obstacle safety check (`FREE` / `BLOCKED`).
4. **Trajectory Model + `PurePursuitController`**: Predicts waypoints and computes steering angle.
5. **`IntersectionDecisionMaker` + `RacecarController`**: 2-phase 90° turns, kick-start throttle boost, and timed distance traversal.

### 1. Setup Environment & Load All 3 AI Models (ONNX Runtime / TensorRT)

In [ ]:
import os
import sys
import cv2
import time
import numpy as np
import onnxruntime as ort
from pathlib import Path

# Package Imports from jetracer_ai
from jetracer_ai.hardware import RacecarController, NvidiaRacecar
from jetracer_ai.utils import CameraStream, bgr8_to_jpeg
from jetracer_ai.core import ONNXEngine, PurePursuitController
from jetracer_ai.urban_traffic import (
    YOLOProcessor,
    RoadProcessor,
    TrafficFSM,
    IntersectionDecisionMaker,
    UrbanObjectDetector
)

# Model Paths (Centralized Relative Paths)
yolo_model_path       = "models/urban_traffic/best.onnx"
collision_model_path  = "models/urban_traffic/best_model_mobilenet.onnx"
trajectory_model_path = "models/urban_traffic/best_trajectory_mobilenet.onnx"
if not os.path.exists(trajectory_model_path):
    trajectory_model_path = "models/urban_traffic/direction_model.onnx"

print(f"[*] Loading Model 1 (YOLO Sign Detector)       : {yolo_model_path}")
print(f"[*] Loading Model 2 (Collision Avoidance)      : {collision_model_path}")
print(f"[*] Loading Model 3 (Conditioned Trajectory)   : {trajectory_model_path}")

# Initialize Inference Engines for All 3 Models
yolo_engine       = ONNXEngine(yolo_model_path, cache_dir="./trt_cache")
collision_engine  = ONNXEngine(collision_model_path, cache_dir="./trt_cache")
trajectory_engine = ONNXEngine(trajectory_model_path, cache_dir="./trt_cache")

print("\n[✓] All 3 AI Models loaded successfully into TensorRT / CUDA Execution Providers!")


### 2. Initialize Processors, Controllers & Traffic FSM

In [ ]:
# 1. Processors
yolo_processor = YOLOProcessor(
    img_size=640,
    conf_thresh=0.45,
    iou_thresh=0.45,
    classes=['green-light', 'left-turn-sign', 'prohibition-sign', 'red-light', 'right-turn-sign', 'straight-ahead-sign']
)
road_processor = RoadProcessor(img_size=(160, 160), threshold=0.5)

# 2. FSM & Decision Maker
fsm = TrafficFSM(
    default_state='FORWARD',
    conf_threshold=0.5,
    min_consecutive_frames=3,
    state_timeout=2.0,
    min_bbox_area=900,
    roi_x_min=0.05,
    roi_x_max=0.95
)

# 3. PurePursuit Controller & Hardware Controller
pure_pursuit = PurePursuitController(lookahead_distance=0.3, gain_k=2.0)
car_controller = RacecarController(
    base_throttle=0.15,
    cm_per_second=30.0,
    kick_throttle=0.5,
    kick_duration=0.2,
    turn90_forward_duration_left=1.2,
    turn90_reverse_duration_left=1.0
)

decision_maker = IntersectionDecisionMaker(
    controller=car_controller,
    min_area_trigger=3500,
    y_bottom_trigger=320,
    cooldown_time=3.0,
    memory_hold_time=0.8
)

# 4. Obstacle Escape State Machine Parameters (Identical to Option A)
CONFIRM_FRAMES = 3
CONFIRM_THRESHOLD = 0.5
maneuver_state = 'DRIVE'
blocked_frame_count = 0
maneuver_start_time = time.time()
actual_steering = 0.0
actual_throttle = 0.0

print("[✓] Processors, FSM, PurePursuit, Decision Maker, and Obstacle Escape Parameters initialized!")


### 3. Realtime Multi-Task Autonomous Loop (3 Models + ROS Camera)

In [ ]:
# Initialize ROS Camera Stream
camera = CameraStream(
    topic_name="/csi_cam_0/image_raw",
    width=640,
    height=480,
    record_video=True,
    output_path="recordings/smart_city_3models_run.avi",
    fps=20
)

print("[*] Starting Option B Autonomous Control Loop (Trajectory Navigation + Option A Obstacle Evasion).")

try:
    while True:
        frame = camera.get_frame()
        if frame is None:
            time.sleep(0.02)
            continue

        now = time.time()

        # --- MODEL 1: YOLO Traffic Sign Detection & FSM Update ---
        yolo_input, orig_h, orig_w = yolo_processor.preprocess(frame)
        yolo_output = yolo_engine.infer(yolo_input)
        detections  = yolo_processor.postprocess(yolo_output, orig_h, orig_w)
        fsm_state   = fsm.update(detections, img_w=orig_w, img_h=orig_h)

        # --- MODEL 2: MobileNet Collision Avoidance Safety Check ---
        collision_input  = road_processor.preprocess(frame)
        collision_output = collision_engine.infer(collision_input)
        road_status      = road_processor.postprocess(collision_output)

        # --- DEBOUNCE ROAD STATUS (SAME AS OPTION A) ---
        blocked_prob = road_status.get('blocked_probability', 0.0)
        if blocked_prob >= CONFIRM_THRESHOLD:
            blocked_frame_count += 1
        else:
            blocked_frame_count = max(0, blocked_frame_count - 1)

        # --- OBSTACLE ESCAPE STATE MACHINE (SAME AS OPTION A) ---
        if maneuver_state == 'DRIVE':
            if blocked_frame_count >= CONFIRM_FRAMES:
                maneuver_state = 'REVERSE_TURNING'
                maneuver_start_time = now
                actual_steering = -0.7
                actual_throttle = -0.22
                car_controller.steering = actual_steering
                car_controller.throttle = actual_throttle
            else:
                # --- LÚC FREE: ĐIỀU HƯỚNG BỞI TRAJECTORY MODEL (5 WAYPOINTS) ---
                cmd_idx = 1
                if fsm_state == fsm.STATE_TURN_LEFT:
                    cmd_idx = 0
                elif fsm_state == fsm.STATE_TURN_RIGHT:
                    cmd_idx = 2

                traj_input = cv2.resize(frame, (224, 224)).astype(np.float32) / 255.0
                traj_input = (traj_input - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
                traj_input = np.transpose(traj_input, (2, 0, 1))
                traj_input = np.expand_dims(traj_input, axis=0)

                try:
                    traj_output = trajectory_engine.session.run(None, {
                        trajectory_engine.input_name: traj_input,
                        trajectory_engine.session.get_inputs()[1].name: np.array([cmd_idx], dtype=np.int64)
                    })[0]
                    waypoints = traj_output.reshape(5, 2)
                    lane_steering = pure_pursuit.compute_steering(waypoints)
                except Exception:
                    lane_steering = 0.0

                decision_status = decision_maker.process_detections(
                    detections=detections,
                    lane_steering=lane_steering,
                    possible_directions=[fsm_state]
                )
                actual_steering = car_controller.steering
                actual_throttle = car_controller.throttle

        elif maneuver_state == 'REVERSE_TURNING':
            if now - maneuver_start_time < 1.2:
                actual_steering = -0.7
                actual_throttle = -0.22
            else:
                maneuver_state = 'PAUSE'
                maneuver_start_time = now
                actual_steering = 0.0
                actual_throttle = 0.0
            car_controller.steering = actual_steering
            car_controller.throttle = actual_throttle

        elif maneuver_state == 'PAUSE':
            if now - maneuver_start_time < 0.3:
                actual_steering = 0.0
                actual_throttle = 0.0
            else:
                maneuver_state = 'CHECK_FORWARD'
                maneuver_start_time = now
                actual_steering = 0.0
                actual_throttle = 0.15
            car_controller.steering = actual_steering
            car_controller.throttle = actual_throttle

        elif maneuver_state == 'CHECK_FORWARD':
            if now - maneuver_start_time < 0.8:
                actual_steering = 0.0
                actual_throttle = 0.15
            else:
                if blocked_frame_count < CONFIRM_FRAMES:
                    maneuver_state = 'DRIVE'
                    blocked_frame_count = 0
                else:
                    maneuver_state = 'REVERSE_TURNING'
                    maneuver_start_time = now
                    actual_steering = -0.7
                    actual_throttle = -0.22
            car_controller.steering = actual_steering
            car_controller.throttle = actual_throttle

        # Draw overlay annotations for video recording & visual debug
        annotated_frame = yolo_processor.draw_bboxes(frame.copy(), detections)
        cv2.putText(annotated_frame, f"FSM: {fsm_state} | Maneuver: {maneuver_state}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        cv2.putText(annotated_frame, f"Road: {road_status['status']} ({blocked_prob:.2f}) | S:{actual_steering:.2f} T:{actual_throttle:.2f}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2)

        time.sleep(0.02)

except KeyboardInterrupt:
    print("\n[*] Autonomous loop stopped by user.")
finally:
    car_controller.stop()
    camera.release()
    print("[✓] Hardware controller stopped & video log saved.")
